<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/LTE_Eng_Rule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [162]:
import pandas as pd

# Raw GitHub file URL
url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE_01_03_2026%20-%20Lite.xlsx"
#url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE%20Master%20Raw%20Data.xlsx"

# Load Excel file
df = pd.read_excel(url)

# Show first few rows
df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,datetime,traffic_load_mbps,Total Power
0,101,10111,11,1,2026-03-01 00:00,11.90,17.28
1,101,10111,11,2,2026-03-01 00:15,12.03,17.28


In [132]:
url1 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Site%20information.xlsx"

# Load into dataframe
info_4g = pd.read_excel(url1)
info_4g.head(2)


,#,Site_ID,Site Name,lbbp_boards,bbu3900,bbu3910,rru_count
0,1,101,AIRPORT_MAHE,2,1,1,12
1,2,102,AIRPORT_PRASLIN,1,1,0,4


In [133]:
df["rru_count"] = 0
df["bbu3900"] = 0
df["bbu3910"] = 0

df["bbu3900_bp"] = 0
df["bbu3910_bp"] = 0

df["lte_traffic_max"] = 0

df["bbu_power_band"] = 0
df["bbu_extra_power"] = 0

df["lbbp_boards_count"] = 0
df["lbbp_bp"] = 0

df["lbbp_power_band"] = 0
df["lbbp_extra_power"] = 0

df["rru_bp"] = 180

df["rruu_power_band"] = 0
df["rru_extra_power"] = 0

df["calc_sec_power"] = 0
df["power_difference"] = 0
df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,datetime,traffic_load_mbps,Total Power,rru_count,bbu3900,bbu3910,...,bbu_extra_power,lbbp_boards_count,lbbp_bp,lbbp_power_band,lbbp_extra_power,rru_bp,rruu_power_band,rru_extra_power,calc_sec_power,power_difference
0,101,10111,11,1,2026-03-01 00:00,11.90,17.28,0,0,0,...,0,0,0,0,0,180,0,0,0,0
1,101,10111,11,2,2026-03-01 00:15,12.03,17.28,0,0,0,...,0,0,0,0,0,180,0,0,0,0


In [134]:
#rru_count map
rru_map = dict(zip(info_4g["Site_ID"], info_4g["rru_count"]))
df["rru_count"] = df["Site_ID"].map(rru_map).fillna(0).astype(int)
df[["Site_ID", "rru_count"]].head(2)

,Site_ID,rru_count
0,101,12
1,101,12


In [135]:
#df.to_excel("rru_count.xlsx", index=False)

In [136]:
bbu3900_map = dict(zip(info_4g["Site_ID"], info_4g["bbu3900"]))
df["bbu3900"] = df["Site_ID"].map(bbu3900_map).fillna(0).astype(int)

bbu3910_map = dict(zip(info_4g["Site_ID"], info_4g["bbu3910"]))
df["bbu3910"] = df["Site_ID"].map(bbu3910_map).fillna(0).astype(int)
df[["Site_ID", "bbu3900", "bbu3910"]].head()

,Site_ID,bbu3900,bbu3910
0,101,1,1
1,101,1,1
2,101,1,1
3,101,1,1
4,101,1,1


In [137]:
#df.to_excel("bbu_map.xlsx", index=False)

In [138]:
import numpy as np
df["bbu3900_bp"] = np.ceil(((df["bbu3900"] * 55) / df["rru_count"]) * 100) / 100
df["bbu3910_bp"] = np.ceil(((df["bbu3910"] * 65) / df["rru_count"]) * 100) / 100

df[["Site_ID", "rru_count", "bbu3900", "bbu3900_bp", "bbu3910", "bbu3910_bp"]].head()

,Site_ID,rru_count,bbu3900,bbu3900_bp,bbu3910,bbu3910_bp
0,101,12,1,4.59,1,5.42
1,101,12,1,4.59,1,5.42
2,101,12,1,4.59,1,5.42
3,101,12,1,4.59,1,5.42
4,101,12,1,4.59,1,5.42


In [139]:
#df.to_excel("bbu_bp.xlsx", index=False)

In [140]:
df["lte_traffic_max"] = np.select(

    [df["traffic_load_mbps"] < 15,
     df["traffic_load_mbps"] < 30,
     df["traffic_load_mbps"] < 45,
     df["traffic_load_mbps"] < 60,
     df["traffic_load_mbps"] < 75,
     df["traffic_load_mbps"] < 100],

    [15, 30, 45, 60, 75, 100],
    default=125
)
df[["traffic_load_mbps", "lte_traffic_max"]].head()

,traffic_load_mbps,lte_traffic_max
0,11.90,15
1,12.03,15
2,12.08,15
3,11.96,15
4,12.48,15


In [141]:
df["bbu_power_band"] = np.select(

    [
        df["traffic_load_mbps"] < 15,
        df["traffic_load_mbps"] < 30,
        df["traffic_load_mbps"] < 45,
        df["traffic_load_mbps"] < 60,
        df["traffic_load_mbps"] < 75,
        df["traffic_load_mbps"] < 100
    ],

    [
        2, 4, 6, 8, 10, 12],

    default=14

)

df[["traffic_load_mbps", "bbu_power_band"]].head()

,traffic_load_mbps,bbu_power_band
0,11.90,2
1,12.03,2
2,12.08,2
3,11.96,2
4,12.48,2


In [142]:
#df.to_excel("bbu_band.xlsx", index=False)

In [143]:
df["bbu_extra_power"] = np.ceil(

    (((df["traffic_load_mbps"] / df["lte_traffic_max"])
      * df["bbu_power_band"]) / df["rru_count"]) * 100) / 100

df[["traffic_load_mbps", "lte_traffic_max", "bbu_power_band", "rru_count", "bbu_extra_power"]].head(2)

,traffic_load_mbps,lte_traffic_max,bbu_power_band,rru_count,bbu_extra_power
0,11.90,15,2,12,0.14
1,12.03,15,2,12,0.14


In [144]:
#df.to_excel("bbu_extra_power.xlsx", index=False)

In [145]:
lbbp_map = dict(zip(info_4g["Site_ID"], info_4g["lbbp_boards"]))
df["lbbp_boards_count"] = df["Site_ID"].map(lbbp_map).fillna(0).astype(int)
df[["Site_ID", "lbbp_boards_count"]].head(2)

,Site_ID,lbbp_boards_count
0,101,2
1,101,2


In [146]:
#df.to_excel("lbbp_boards.xlsx", index=False)

In [147]:
df["lbbp_bp"] = np.ceil(((df["lbbp_boards_count"] * 42.5) / df["rru_count"]) * 100) / 100
df[["lbbp_boards_count", "rru_count", "lbbp_bp"]].head(2)

,lbbp_boards_count,rru_count,lbbp_bp
0,2,12,7.09
1,2,12,7.09


In [148]:
#df.to_excel("lbbp_bp.xlsx", index=False)

In [149]:
#output_df = df[["Site_ID","Cell_ID","lbbp_bp"]];output_df.to_excel("Filtered_Output.xlsx", index=False)

In [150]:
df["lbbp_power_band"] = np.select(

    [df["traffic_load_mbps"] < 15,
     df["traffic_load_mbps"] < 30,
     df["traffic_load_mbps"] < 45,
     df["traffic_load_mbps"] < 60,
     df["traffic_load_mbps"] < 75,
     df["traffic_load_mbps"] < 100],

    [3, 6, 9, 12, 15, 18],

    default=21
)

df[["traffic_load_mbps", "lbbp_power_band"]].head(2)

,traffic_load_mbps,lbbp_power_band
0,11.90,3
1,12.03,3


In [151]:
df["lbbp_extra_power"] = np.ceil(

    (((df["traffic_load_mbps"] / df["lte_traffic_max"])
      * df["lbbp_power_band"]) / df["rru_count"]) * 100

) / 100

df[["traffic_load_mbps", "lte_traffic_max",
    "lbbp_power_band", "rru_count",
    "lbbp_extra_power"]].head(2)

,traffic_load_mbps,lte_traffic_max,lbbp_power_band,rru_count,lbbp_extra_power
0,11.90,15,3,12,0.20
1,12.03,15,3,12,0.21


In [152]:
#output_df = df[["Site_ID","Cell_ID","lbbp_extra_power"]];output_df.to_excel("Filtered_Output.xlsx", index=False)

In [153]:
df["rruu_power_band"] = np.select(

    [df["traffic_load_mbps"] < 15,
     df["traffic_load_mbps"] < 30,
     df["traffic_load_mbps"] < 45,
     df["traffic_load_mbps"] < 60,
     df["traffic_load_mbps"] < 75,
     df["traffic_load_mbps"] < 100],

    [10, 20, 30, 40, 50, 55],

    default=60
)

df[["traffic_load_mbps", "rruu_power_band"]].head(2)

,traffic_load_mbps,rruu_power_band
0,11.90,10
1,12.03,10


In [154]:
df["rru_extra_power"] = np.ceil(

    ((df["traffic_load_mbps"] / df["lte_traffic_max"])
     * df["rruu_power_band"]) * 100

) / 100

df[["traffic_load_mbps", "lte_traffic_max",
    "rruu_power_band", "rru_extra_power"]].head()

,traffic_load_mbps,lte_traffic_max,rruu_power_band,rru_extra_power
0,11.90,15,10,7.94
1,12.03,15,10,8.02
2,12.08,15,10,8.06
3,11.96,15,10,7.98
4,12.48,15,10,8.32


In [155]:
#output_df = df[["Site_ID","Cell_ID","rru_extra_power"]];output_df.to_excel("Filtered_Output.xlsx", index=False)

In [160]:
print(df.columns)

Index(['Site_ID', 'Cell_ID', 'Sector_ID', 'trigger_ID', 'datetime',
       'traffic_load_mbps', 'Total Power', 'rru_count', 'bbu3900', 'bbu3910',
       'bbu3900_bp', 'bbu3910_bp', 'lte_traffic_max', 'bbu_power_band',
       'bbu_extra_power', 'lbbp_boards_count', 'lbbp_bp', 'lbbp_power_band',
       'lbbp_extra_power', 'rru_bp', 'rruu_power_band', 'rru_extra_power',
       'calc_sec_power', 'power_difference'],
      dtype='object')


In [157]:
df["calc_sec_power"] = (

    df["bbu3900_bp"] + df["bbu3910_bp"] + df["bbu_extra_power"] + df["lbbp_bp"] + df["lbbp_extra_power"] +
    df["rru_bp"] + df["rru_extra_power"]

)

df[["calc_sec_power"]].head(2)

,calc_sec_power
0,205.38
1,205.47


In [158]:
output_df = df[["Site_ID","calc_sec_power","bbu_extra_power"]];output_df.to_excel("Filtered_Output.xlsx", index=False)

In [159]:
print("Done")

Done
